# BLOQUE 0 — Librerías y configuración global (IMPORTS ÚNICOS)

In [2]:
import os
import re
from pathlib import Path
from getpass import getpass
from functools import lru_cache

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# -----------------------
# Config global / semillas
# -----------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("/content/data")
OUT_DIR = Path("/content/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_3 = {0: "normal", 1: "bache", 2: "severo"}

ModuleNotFoundError: No module named 'pandas'

# BLOQUE 1 — Instalación de dependencias (Colab)

In [ ]:
!pip -q install kaggle
!pip -q install tensorflow

# BLOQUE 2 — Autenticación Kaggle (token por getpass)

In [ ]:

os.environ["KAGGLE_USERNAME"] = "Andres_Vallejo1004"


os.environ["KAGGLE_API_TOKEN"] = getpass("Enter Kaggle API Token: ")


kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)

kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(
    f'{{"username":"{os.environ["KAGGLE_USERNAME"]}","key":"{os.environ["KAGGLE_API_TOKEN"]}"}}',
    encoding="utf-8"
)

!chmod 600 /root/.kaggle/kaggle.json

# BLOQUE 3 — Descarga del dataset y verificación de estructura

In [ ]:
!mkdir -p /content/data
!kaggle datasets download -d shashwatwork/cyclist-accident-prevention-dataset -p /content/data --unzip
!ls -lah /content/data
!ls -R /content/data

# BLOQUE 4 — Indexador del dataset

In [ ]:
def build_index(data_dir: Path) -> pd.DataFrame:
    rows = []
    for route_dir in sorted([p for p in data_dir.iterdir() if p.is_dir()]):
        route = route_dir.name
        for lap_dir in sorted([p for p in route_dir.iterdir() if p.is_dir()]):
            lap = lap_dir.name
            files = [f for f in lap_dir.iterdir() if f.is_file()]
            names = {f.name.lower(): f for f in files}

            def pick_any(patterns):
                for name, path in names.items():
                    for pat in patterns:
                        if re.search(pat, name):
                            return path
                return None

            acc = pick_any([r"accelerometer"])
            gyro = pick_any([r"gyroscope"])
            mag  = pick_any([r"magnetometer"])
            gps  = pick_any([r"_gps_", r"gps"])  # robusto

            rows.append({
                "route": route,
                "lap": lap,
                "acc_path": str(acc) if acc else None,
                "gyro_path": str(gyro) if gyro else None,
                "gps_path": str(gps) if gps else None,
                "mag_path": str(mag) if mag else None,
            })

    df = pd.DataFrame(rows)

    # sanity checks
    assert df["acc_path"].notna().all(), "Faltan accelerometer en algunas filas"
    assert df["gyro_path"].notna().all(), "Faltan gyroscope en algunas filas"
    return df


idx = build_index(DATA_DIR)
idx.head(), idx.shape

# BLOQUE 5 — IO robusto

In [ ]:
def resolve_path(path: str) -> Path:
    p = Path(path)
    if p.exists():
        return p
    if str(p).endswith(".csv.csv"):
        p2 = Path(str(p)[:-4])
        if p2.exists():
            return p2
    return p


def smart_read_raw(path: str) -> pd.DataFrame:
    p = resolve_path(path)
    if not p.exists():
        raise FileNotFoundError(f"No existe: {p}")

    # 1) intento con separadores comunes
    for sep in [";", ",", "\t", "|"]:
        try:
            df = pd.read_csv(p, sep=sep, header=None, engine="python")
            if df.shape[1] > 1:
                return df
        except Exception:
            pass

    # 2) fallback: una sola columna y split manual por separador más frecuente
    df1 = pd.read_csv(p, header=None, engine="python")
    if df1.shape[1] == 1:
        s = df1.iloc[:, 0].astype(str)
        seps = [";", ",", "\t", "|"]
        counts = {sep: s.str.count(sep).mean() for sep in seps}
        best_sep = max(counts, key=counts.get)
        if counts[best_sep] > 0:
            return s.str.split(best_sep, expand=True)

    return df1


def load_sensor_xyz(path: str, prefix: str) -> pd.DataFrame:
    df = smart_read_raw(path)

    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(axis=1, how="all")
    if df.shape[1] == 0:
        raise ValueError(f"Archivo vacío o ilegible: {path}")

    ts_col = df.columns[0]
    df = df.dropna(subset=[ts_col]).copy()
    df = df.rename(columns={ts_col: "timestamp"})

    cols = list(df.columns)

    if len(cols) >= 5:
        xcol, ycol, zcol = cols[2], cols[3], cols[4]
    elif len(cols) >= 4:
        xcol, ycol, zcol = cols[1], cols[2], cols[3]
    else:
        raise ValueError(f"Archivo con pocas columnas ({len(cols)}): {path}")

    df = df.rename(columns={xcol: f"{prefix}x", ycol: f"{prefix}y", zcol: f"{prefix}z"})
    out = df[["timestamp", f"{prefix}x", f"{prefix}y", f"{prefix}z"]].dropna().copy()
    out = out.sort_values("timestamp").reset_index(drop=True)
    return out

# BLOQUE 6 — Merge IMU por lap

In [ ]:
def load_lap_imu(row: pd.Series) -> pd.DataFrame:
    acc = load_sensor_xyz(row["acc_path"], "a")
    gyro = load_sensor_xyz(row["gyro_path"], "g")

    acc["timestamp"] = acc["timestamp"].astype(np.float64)
    gyro["timestamp"] = gyro["timestamp"].astype(np.float64)

    acc = acc.sort_values("timestamp").reset_index(drop=True)
    gyro = gyro.sort_values("timestamp").reset_index(drop=True)

    imu = pd.merge_asof(acc, gyro, on="timestamp", direction="nearest").dropna()

    imu["route"] = row["route"]
    imu["lap"] = row["lap"]
    imu["lap_id"] = f"{row['route']}::{row['lap']}"
    return imu[["timestamp","ax","ay","az","gx","gy","gz","route","lap","lap_id"]]


imu0 = load_lap_imu(idx.iloc[0])
imu0.head(), imu0.shape

# BLOQUE 7 — Ventaneo + split por rutas

In [ ]:
def build_windows_one_lap(imu: pd.DataFrame, win_len=128, stride=64):
    X = imu[["ax","ay","az","gx","gy","gz"]].values.astype(np.float32)
    n = X.shape[0]
    windows, starts = [], []
    for s in range(0, n - win_len + 1, stride):
        windows.append(X[s:s+win_len])
        starts.append(s)
    return np.stack(windows, axis=0), np.array(starts, dtype=int)

# --- FIX: Define X_all and meta by processing all laps ---
all_windows = []
meta_list = []

for i, row in idx.iterrows():
    # Load and merge IMU for this lap
    imu_lap = load_lap_imu(row)
    # Generate windows
    win_lap, _ = build_windows_one_lap(imu_lap)

    all_windows.append(win_lap)
    # Create metadata for each window generated in this lap
    for _ in range(len(win_lap)):
        meta_list.append({"route": row["route"], "lap": row["lap"]})

X_all = np.concatenate(all_windows, axis=0)
meta = pd.DataFrame(meta_list)
# --------------------------------------------------------

train_mask = meta["route"].isin(["First route", "Second route"]).values
val_mask   = meta["route"].isin(["Third route"]).values

X_train = X_all[train_mask]
X_val   = X_all[val_mask]

# BLOQUE 8 — Features + scores

In [ ]:
def compute_window_features_v4(Xw: np.ndarray) -> dict:
    ax, ay, az, gx, gy, gz = Xw[:,0], Xw[:,1], Xw[:,2], Xw[:,3], Xw[:,4], Xw[:,5]
    a_mag = np.sqrt(ax**2 + ay**2 + az**2)
    g_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    return {
        "ax_p05": float(np.percentile(ax, 5)),
        "az_abs_p99": float(np.percentile(np.abs(az), 99)),
        "a_mag_p99": float(np.percentile(a_mag, 99)),
        "g_mag_p99": float(np.percentile(g_mag, 99)),
        "a_mag_std": float(np.std(a_mag)),
        "g_mag_std": float(np.std(g_mag)),
    }


def compute_scores_from_feats(feats: dict) -> tuple[float, float]:
    severity_score = (
        abs(feats["ax_p05"]) +
        feats["a_mag_p99"] +
        feats["g_mag_p99"] +
        feats["g_mag_std"]
    )
    bump_score = feats["az_abs_p99"]
    return float(severity_score), float(bump_score)

# BLOQUE 9 — Etiquetado 3 clases + class_weight + guardado

In [ ]:
# 1) Features y scores por ventana
feat_rows = []
N = X_all.shape[0]
for i in range(N):
    feats = compute_window_features_v4(X_all[i])
    sev_s, bump_s = compute_scores_from_feats(feats)
    feat_rows.append({"i": i, **feats, "severity_score": sev_s, "bump_score": bump_s})

feat_df = pd.DataFrame(feat_rows)

# 2) Calibración con TRAIN
TARGET_SEVERE_RATE = 0.10
TARGET_BUMP_RATE   = 0.25

train_idx = np.where(train_mask)[0]

sev_th = float(np.percentile(
    feat_df.loc[train_idx, "severity_score"].values,
    100 * (1 - TARGET_SEVERE_RATE)
))

bump_th = float(np.percentile(
    feat_df.loc[train_idx, "bump_score"].values,
    100 * (1 - TARGET_BUMP_RATE)
))

print("Thresholds calibrados (TRAIN):")
print(" - sev_th =", sev_th, " (TARGET_SEVERE_RATE=", TARGET_SEVERE_RATE, ")")
print(" - bump_th =", bump_th, " (TARGET_BUMP_RATE  =", TARGET_BUMP_RATE, ")")

# 3) Etiquetado final
labels_3 = np.zeros((N,), dtype=np.int32)
sev_scores  = feat_df["severity_score"].values
bump_scores = feat_df["bump_score"].values

labels_3[sev_scores >= sev_th] = 2
labels_3[(labels_3 == 0) & (bump_scores >= bump_th)] = 1

# Meta etiquetada
meta_labeled_3 = meta.copy().reset_index(drop=True)
meta_labeled_3["label_id"] = labels_3
meta_labeled_3["label_name"] = meta_labeled_3["label_id"].map(LABELS_3)
meta_labeled_3["split"] = np.where(train_mask, "train", np.where(val_mask, "val", "other"))

print("\nDistribución global:")
print(meta_labeled_3["label_name"].value_counts())

print("\nDistribución por split:")
print(meta_labeled_3.groupby("split")["label_name"].value_counts())

# 4) Split final
X_train = X_all[train_mask]
y_train = labels_3[train_mask]

X_val = X_all[val_mask]
y_val = labels_3[val_mask]

meta_train = meta_labeled_3[train_mask].reset_index(drop=True)
meta_val   = meta_labeled_3[val_mask].reset_index(drop=True)

print("\nShapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)

# 5) class_weight
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes.tolist(), weights.tolist()))

print("\nclass_weight:", class_weight)
print("labels:", {int(k): LABELS_3[int(k)] for k in class_weight.keys()})

# 6) Guardar artefactos
np.save(OUT_DIR / "labels_3class.npy", labels_3)
np.save(OUT_DIR / "X_train.npy", X_train)
np.save(OUT_DIR / "y_train.npy", y_train)
np.save(OUT_DIR / "X_val.npy", X_val)
np.save(OUT_DIR / "y_val.npy", y_val)

meta_labeled_3.to_csv(OUT_DIR / "meta_labeled_3class.csv", index=False)
meta_train.to_csv(OUT_DIR / "meta_train_3class.csv", index=False)
meta_val.to_csv(OUT_DIR / "meta_val_3class.csv", index=False)
feat_df.to_csv(OUT_DIR / "window_features_scores_3class.csv", index=False)

with open(OUT_DIR / "thresholds_3class.txt", "w", encoding="utf-8") as f:
    f.write(f"TARGET_SEVERE_RATE={TARGET_SEVERE_RATE}\n")
    f.write(f"TARGET_BUMP_RATE={TARGET_BUMP_RATE}\n")
    f.write(f"sev_th={sev_th}\n")
    f.write(f"bump_th={bump_th}\n")

print("\nGuardado en:", OUT_DIR)

# BLOQUE 10 — tf.data pipeline

In [ ]:
BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

ds_train = tf.data.Dataset.from_tensor_slices((X_train, y_train))
ds_train = ds_train.shuffle(min(len(X_train), 20000), seed=SEED, reshuffle_each_iteration=True)
ds_train = ds_train.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_val = tf.data.Dataset.from_tensor_slices((X_val, y_val))
ds_val = ds_val.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# BLOQUE 11 — Modelo CNN 1D

In [ ]:
def build_cnn_1d_classifier(T=128, C=6, n_classes=3) -> keras.Model:
    inp = keras.Input(shape=(T, C))

    x = layers.Conv1D(32, 7, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Conv1D(64, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Conv1D(128, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    return keras.Model(inp, out)


T, C = X_train.shape[1], X_train.shape[2]
model = build_cnn_1d_classifier(T=T, C=C, n_classes=3)
model.summary()

# BLOQUE 12 — Compile + fit

In [ ]:
LR = 1e-3

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

EPOCHS = 60
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)